# RPS PreTrained YOLO Fine-tuning

## g-drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## 경로 설정

In [ ]:
# 다른 경로에 저장하려면, 아래 경로를 수정하세요
save_dir = '/content/drive/MyDrive/files/SmartCheckout/model/'
dataset_zip = '/content/drive/MyDrive/files/SmartCheckout/image_set.zip'
dataset_root = '/content/image_set'
project_root = '/content/rps_runs'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## RPS 데이터셋 준비

In [ ]:
!unzip -q -o {dataset_zip}

## Ultralytics 설치 및 import

In [ ]:
!pip -q install -U ultralytics==8.4.98

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.1 MB/s eta 0:00:00


## 학습

In [ ]:
DATA = f'{dataset_root}/data.yaml'
  #imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  #imgsz=480 epochs=100 batch=16 device=0 workers=4 \
#!yolo detect train model=yolo11n.pt data={DATA} \
#  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
#  project={project_root} degrees=15.0 flipud=0.5 \
# fliplr=0.5 mosaic=1.0 name=rps_yolo11n
!yolo detect train model=yolo11n.pt data={DATA} \
  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  project={project_root} degrees=90.0 flipud=0.5 \
  hsv_v=0.4 hsv_s=0.3 hsv_h=0.015 mosaic=1.0 \
  fliplr=0.5 mosaic=1.0 name=rps_yolo11n

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.164 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/image_set/data.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, era

## LiteRT로 변환

In [ ]:
BEST = f'{project_root}/rps_yolo11n/weights/best.pt'

# 양자화 안함
!yolo export model={BEST} format=litert imgsz=320 data={DATA}

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 101 layers, 2,583,907 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 13, 4725) (5.2 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 543ms
Prepared 15 packages in 1.59s
Uninstalled 3 packages in 36ms
Installed 15 packages in 39ms
 + ai-edge-litert==2.2.0
 + ai-edge-quantizer==0.9.0
 + backports-strenum==1.3.1
 + fire==0.7.1
 - immutabledict==4.3.1
 + immutabledict==4.2.1
 + jaxtyping==0.3.11
 + litert-converter==0.4.0
 + litert-lm-builder==0.17.1
 + litert-torch==0.9.4
 + o

In [ ]:
# PTQ(INT8) (입력과 출력은 float32)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=8

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,583,907 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 13, 4725) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/image_set/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1951.1±473.5 MB/s, size: 78.3 KB)
val: Scanning /content/image_set/labels/test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 22.9Mit/s 0.0s
WARNING ⚠️ LiteRT: >300 images recommended for INT8 calibration, found 180 images.
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /u

In [ ]:
# PTQ(Dynamic Range)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=w8a32

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,583,907 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n/weights/best.pt' with input shape (1, 3, 480, 480) BCHW and output shape(s) (1, 13, 4725) (5.2 MB)
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

LiteRT: starting export with litert_torch 0.9.4...
(00:00) [START] LiteRT-Torch Convert
(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default
(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > 
ExportedProgram Run Decompositions
/u

## tflite 파일 저장

In [ ]:
!cp {project_root}/rps_yolo11n/weights/*.tflite {save_dir}

In [ ]:
# 1. PyTorch 원본 모델 검증
!yolo val task=detect model={project_root}/rps_yolo11n/weights/best.pt data={DATA} imgsz=320

# 2. FP32 LiteRT 모델 검증
!yolo val task=detect model={project_root}/rps_yolo11n/weights/best.tflite data={DATA} imgsz=320

# 3. INT8 LiteRT 모델 검증 (생성된 int8 파일 경로 지정)
!yolo val task=detect model={project_root}/rps_yolo11n/weights/best_int8.tflite data={DATA} imgsz=320

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,583,907 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2267.3±284.7 MB/s, size: 96.0 KB)
val: Scanning /content/image_set/labels/test.cache... 180 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 180/180 29.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 6.0it/s 2.0s
                   all        180        220      0.937      0.921      0.933      0.614
                Coffee         21         21      0.991          1      0.995      0.695
                 Water         21         21       0.98          1      0.995      0.694
                 Letbe         29         29      0.982          1      0.995      0.608
                  Cola         28         28      0.993          1      0.995      0.713
              Pocachip         29         29 